___


___

# Tokenización
El primer paso para crear un objeto `Doc` es dividir el texto entrante en piezas componentes o "tokens".

In [1]:
# Importar spaCy y cargar la biblioteca de idiomas
import spacy
nlp = spacy.load('en_core_web_sm')

In [2]:
# Cree una cadena que incluya comillas de apertura y cierre
mystring = '"We\'re moving to L.A.!"'
print(mystring)

"We're moving to L.A.!"


In [3]:
# Cree un objeto Doc y explore tokens
doc = nlp(mystring)

for token in doc:
    print(token.text, end=' | ')

" | We | 're | moving | to | L.A. | ! | " | 

<img src="tokenization.png" width="600">

- **Prefijo**: Carácter(es) al principio &#9656; `$ ( “ ¿`
- **Sufijo**: Carácter(es) al final &#9656; `km ) , . ! ”`
- **Infijo**: Carácter(es) entre &#9656; `- -- / ...`
- **Excepción**: regla de caso especial para dividir una cadena en varios tokens o evitar que un token se divida cuando se aplican las reglas de puntuación &#9656; 'St. U.S.

Tenga en cuenta que las fichas son partes del texto original. Es decir, no vemos ninguna conversión a raíces de palabras o lemas (formas básicas de las palabras) y no hemos visto nada sobre organizaciones/lugares/dinero, etc. Los tokens son los componentes básicos de un objeto Doc: todo lo que ayuda Entendamos que el significado del texto se deriva de las fichas y su relación entre sí.

## Prefijos, Sufijos e Infijos
spaCy aislará la puntuación que *no* forma parte integral de una palabra. A las comillas, comas y signos de puntuación al final de una oración se les asignará su propio token. Sin embargo, la puntuación que existe como parte de una dirección de correo electrónico, un sitio web o un valor numérico se mantendrá como parte del token.

In [4]:
doc2 = nlp(u"We're here to help! Send snail-mail, email support@oursite.com or visit us at http://www.oursite.com!")

for t in doc2:
    print(t)

We
're
here
to
help
!
Send
snail
-
mail
,
email
support@oursite.com
or
visit
us
at
http://www.oursite.com
!


<font color=green>Tenga en cuenta que los signos de exclamación, la coma y el guión en 'correo postal' tienen asignados sus propios tokens, pero se conservan tanto la dirección de correo electrónico como el sitio web.</font>

In [5]:
doc3 = nlp(u'A 5km NYC cab ride costs $10.30') ## u: cadena unicode

for t in doc3:
    print(t)

A
5
km
NYC
cab
ride
costs
$
10.30


<font color=green>Aquí, a la unidad de distancia y al signo de dólar se les asignan sus propios tokens, pero se conserva la cantidad en dólares.</font>

## Excepciones
La puntuación que exista como parte de una abreviatura conocida se mantendrá como parte del token.

In [6]:
doc4 = nlp(u"Let's visit St. Louis in the U.S. next year.")

for t in doc4:
    print(t)

Let
's
visit
St.
Louis
in
the
U.S.
next
year
.


<font color=green>Aquí se conservan las abreviaturas de "Santo" y "Estados Unidos".</font>

## Conteo de Tokens
Los objetos `Doc` tienen un número determinado de tokens:

In [7]:
len(doc)

8

## Contar entradas de vocabulario
¡Los objetos `Vocab` contienen una biblioteca completa de elementos!

In [8]:
len(doc.vocab)

57852

<font color=green>NOTA: este número cambia en función de la biblioteca de idiomas cargada al principio y de cualquier léxico nuevo introducido en el `vocabulario` cuando se creó el `Doc`.</font>

## Los tokens se pueden recuperar por posición de índice y segmento
Los objetos `Doc` se pueden considerar como listas de objetos `token`. Como tal, los tokens individuales se pueden recuperar por posición de índice, y los intervalos de tokens se pueden recuperar a través de cortes:

In [9]:
doc5 = nlp(u'It is better to give than to receive.')

# Recuperar el tercer token:
doc5[2]

better

In [10]:
# Recupera tres tokens del medio:
doc5[2:5]

better to give

In [11]:
# Recuperar los últimos cuatro tokens:
doc5[-4:]

than to receive.

## Los tokens no se pueden reasignar
Aunque los objetos `Doc` pueden considerarse listas de tokens, *no* admiten la reasignación de elementos:

In [12]:
doc6 = nlp(u'My dinner was horrible.')
doc7 = nlp(u'Your dinner was delicious.')

In [13]:
# Intenta cambiar "My dinner was horrible" a "My dinner was delicious"
doc6[3] = doc7[3]

TypeError: 'spacy.tokens.doc.Doc' object does not support item assignment

___
# Entidades Nombradas
Yendo un paso más allá de los tokens, las *entidades nombradas* agregan otra capa de contexto. El modelo de lenguaje reconoce que ciertas palabras son nombres de organizaciones mientras que otras son ubicaciones, y otras combinaciones se relacionan con dinero, fechas, etc. Las entidades nombradas son accesibles a través de la propiedad `ents` de un objeto `Doc`.

In [14]:
doc8 = nlp(u'Apple to build a Hong Kong factory for $6 million')

for token in doc8:
    print(token.text, end=' | ')

print('\n----')

for ent in doc8.ents:
    print(ent.text+' - '+ent.label_+' - '+str(spacy.explain(ent.label_)))

Apple | to | build | a | Hong | Kong | factory | for | $ | 6 | million | 
----
Apple - ORG - Companies, agencies, institutions, etc.
Hong Kong - GPE - Countries, cities, states
$6 million - MONEY - Monetary values, including unit


<font color=green>Observe cómo dos tokens se combinan para formar la entidad `Hong Kong`, y tres tokens se combinan para formar la entidad monetaria: `$6 millones`</font>

In [15]:
len(doc8.ents)

3

El reconocimiento de entidades nombradas (NER) es una importante herramienta de aprendizaje automático aplicada al procesamiento del lenguaje natural.<br>Haremos mucho más con ella en una próxima sección. Para obtener más información sobre **entidades nombradas**, visite https://spacy.io/usage/linguistic-features#named-entities

---
# Fragmentos de sustantivos
Similar a `Doc.ents`, `Doc.noun_chunks` son otra propiedad de objeto. *Los fragmentos de sustantivos* son "frases nominales base": frases planas que tienen un sustantivo como encabezado. Puede pensar en fragmentos de sustantivos como un sustantivo más las palabras que describen el sustantivo; por ejemplo, en [la canción de Sheb Wooley de 1958] (https://en.wikipedia.org/wiki/The_Purple_People_Eater), un *"tuerto, un cornudo, volador, devorador púrpura de personas"* sería un fragmento de sustantivo largo.

In [16]:
doc9 = nlp(u"Autonomous cars shift insurance liability toward manufacturers.")

for chunk in doc9.noun_chunks:
    print(chunk.text)

Autonomous cars
insurance liability
manufacturers


In [17]:
doc10 = nlp(u"Red cars do not carry higher insurance rates.")

for chunk in doc10.noun_chunks:
    print(chunk.text)

Red cars
higher insurance rates


In [18]:
doc11 = nlp(u"He was a one-eyed, one-horned, flying, purple people-eater.")

for chunk in doc11.noun_chunks:
    print(chunk.text)

He
a one-eyed, one-horned, flying, purple people-eater


Veremos componentes adicionales de noun_chunks además de `.text` en una próxima sección.<br>Para obtener más información sobre **noun_chunks**, visite https://spacy.io/usage/linguistic-features#noun-chunks

___
# Visualizadores incorporados

spaCy incluye una herramienta de visualización integrada llamada **displaCy**. displaCy puede detectar si está trabajando en un cuaderno Jupyter y devolverá un marcado que se puede representar en una celda de inmediato. Cuando exporte su cuaderno, las visualizaciones se incluirán como HTML.

Para obtener más información, visite https://spacy.io/usage/visualizers

## Visualizando el análisis de dependencia
Ejecute la celda a continuación para importar la visualización y mostrar el gráfico de dependencia

In [19]:
from spacy import displacy

doc = nlp(u'Apple is going to build a U.K. factory for $6 million.')
displacy.render(doc, style='dep', jupyter=True, options={'distance': 110})

El argumento `'distance'` opcional establece la distancia entre tokens. Si la distancia se hace demasiado pequeña, el texto que aparece debajo de las flechas cortas puede quedar demasiado comprimido para leer.

## Visualizando el reconocedor de entidades

In [20]:
doc = nlp(u'Over the last quarter Apple sold nearly 20 thousand iPods for a profit of $6 million.')
displacy.render(doc, style='ent', jupyter=True)

___
## Creación de visualizaciones fuera de Jupyter
Si está utilizando otro IDE de Python o está escribiendo un script, puede elegir que spaCy sirva html por separado:

In [23]:
doc = nlp(u'This is a sentence.')
displacy.serve(doc, style='dep', port=8000)


    Serving on port 8000...
    Using the 'dep' visualizer



127.0.0.1 - - [14/Apr/2023 22:04:56] "GET / HTTP/1.1" 200 3057
127.0.0.1 - - [14/Apr/2023 22:04:56] "GET /favicon.ico HTTP/1.1" 200 3057
127.0.0.1 - - [14/Apr/2023 22:05:59] "GET / HTTP/1.1" 200 3057
127.0.0.1 - - [14/Apr/2023 22:05:59] "GET /favicon.ico HTTP/1.1" 200 3057



    Shutting down server on port 8000.



<font color=blue>**Después de ejecutar la celda anterior, haga clic en el enlace a continuación para ver el análisis de dependencia**:</font>

http://127.0.0.1:8000
<br><br>
<font color=red>**Para apagar el servidor y volver a jupyter**, interrumpa el kernel a través del menú **Kernel** de arriba, presionando el cuadrado negro en la barra de herramientas o escribiendo el atajo de teclado ` Esc`, `I`, `I`</font>

¡Excelente! Ahora debería comprender cómo la tokenización divide el texto en elementos individuales, cómo las entidades nombradas brindan contexto y cómo ciertas herramientas ayudan a visualizar las reglas gramaticales y las etiquetas de las entidades.
## A continuación: Stemming